### Build a Simple LLM Application with LCEL
In this quickstart we'll show you how to build a simple LLM application with LangChain. This application will translate text from English into another language. This is a relatively simple LLM application - it's just a single LLM call plus some prompting. Still, this is a great way to get started with LangChain - a lot of features can be built with just some prompting and an LLM call!

After seeing this video, you'll have a high level overview of:

- Using language models

- Using PromptTemplates and OutputParsers

- Using LangChain Expression Language (LCEL) to chain components together

- Debugging and tracing your application using LangSmith

- Deploying your application with LangServe

In [1]:
## OpenAI API Key and Open Source models --llama3,gemma2,mistral,groq

import os
from dotenv import load_dotenv
load_dotenv()

import openai
openai.api_key = os.getenv('OPENAI_API_KEY')


groq_api_key = os.getenv("GROQ_API_KEY")
groq_api_key

'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'

In [2]:
!pip install langchain_groq

In [6]:
from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq
model = ChatGroq(model="openai/gpt-oss-120b",groq_api_key=groq_api_key)
model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x1338a9cf0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x1338a8d60>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [4]:
!pip install langchain_core

In [9]:
from langchain_core.messages import HumanMessage,SystemMessage

messages = [
    SystemMessage(content='Translate the following from English to French'),
    HumanMessage(content="Hello How are you")
]

result = model.invoke(messages)

In [10]:
result

AIMessage(content='Bonjour, comment ça va\u202f?', additional_kwargs={'reasoning_content': 'The user wants translation English to French: "Hello How are you". Should translate: "Bonjour, comment ça va ?" Possibly "Bonjour, comment vas-tu ?" but formal: "Bonjour, comment allez-vous ?" The phrase "Hello How are you" likely informal: "Bonjour, comment ça va ?" We\'ll output translation.'}, response_metadata={'token_usage': {'completion_tokens': 81, 'prompt_tokens': 85, 'total_tokens': 166, 'completion_time': 0.166891593, 'completion_tokens_details': {'reasoning_tokens': 65}, 'prompt_time': 0.003199405, 'prompt_tokens_details': None, 'queue_time': 0.051077544, 'total_time': 0.170090998}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_8a618bed98', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019dcd9b-b067-7253-a34a-ea0b51d49a5a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 85, 'outp

In [11]:
from langchain_core.output_parsers import StrOutputParser
parser = StrOutputParser()
parser.invoke(result)

'Bonjour, comment ça va\u202f?'

In [12]:
# USing LCEL - we can chain the components
chain = model|parser
chain.invoke(messages)

'Bonjour, comment ça va\u202f?'

In [14]:
## Prompt templates
from langchain_core.prompts import ChatPromptTemplate
generic_template = "Translate the following into {language}:"
prompt = ChatPromptTemplate.from_messages(
        [("system",generic_template),("user",'{text}')]
)

In [16]:
result = prompt.invoke({"language":"French","text":"Hello"})

In [17]:
result.to_messages()

[SystemMessage(content='Translate the following into French:', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello', additional_kwargs={}, response_metadata={})]

In [18]:
chain = prompt|model|parser
chain.invoke({'language':'french','text':'hello'})

'Bonjour.'